# Lección 10 - Agentes de IA en Producción

En esta lección aprenderás **patrones de producción** para agentes de IA usando el Microsoft Agent Framework (Python). Cubrimos:

- **Observabilidad** — añadir medición de tiempos y registro a las interacciones del agente
- **Evaluación** — usar un agente evaluador para puntuar la calidad de las respuestas
- **Gestión de costos** — estrategias para optimización de tokens y selección de modelo

El escenario es un **agente de viajes** que ayuda a los usuarios a planificar viajes, con monitoreo y evaluación integrados.


## Configuración


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## Consideraciones de Producción

Mover agentes de IA de prototipos a producción requiere prestar atención cuidadosa a tres pilares:

1. **Observability** — Necesitas visibilidad de lo que hace el agente, cuánto tiempo tarda y qué herramientas invoca. Sin trazado y registro, depurar problemas en producción es casi imposible.

2. **Evaluation** — Las comprobaciones de calidad automatizadas garantizan que las respuestas del agente se mantengan precisas, completas y útiles con el tiempo. Un agente evaluador puede puntuar las respuestas según criterios definidos.

3. **Cost Management** — El uso de tokens impacta directamente en el costo. Estrategias como la optimización de prompts, la selección de modelos y el almacenamiento en caché ayudan a mantener los gastos bajo control sin sacrificar la calidad.


## Construyendo un agente observable

Definimos herramientas de viaje y envolvemos la llamada al agente con medición de tiempo para que podamos supervisar la latencia. En producción, integrarías OpenTelemetry u otro backend de trazado similar.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Patrones de evaluación

Un patrón común en producción es usar un segundo agente como **evaluador**. El evaluador puntúa la respuesta del agente principal según criterios predefinidos como completitud, exactitud y utilidad.

Esto permite:
- Pasarelas de control de calidad automatizadas antes de que las respuestas lleguen a los usuarios
- Detección de regresiones cuando cambian las indicaciones o los modelos
- Monitoreo continuo del rendimiento del agente a lo largo del tiempo


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Estrategias de gestión de costos

Controlar los costos es crítico para los agentes de IA en producción. Aquí están las estrategias clave:

| Strategy | Description |
|---|---|
| **Optimización de prompts** | Mantén las instrucciones del sistema concisas. Elimina el contexto redundante para reducir los tokens de entrada. |
| **Selección de modelo** | Utiliza modelos más pequeños y económicos (p. ej., GPT-4o-mini) para tareas simples como clasificación o extracción, y reserva modelos más grandes para razonamiento complejo. |
| **Caché** | Almacena en caché los resultados de las herramientas y las consultas frecuentes para evitar llamadas a la API redundantes. |
| **Presupuestos de tokens** | Establece límites de `max_tokens` para evitar respuestas inesperadamente largas. |
| **Agrupación** | Agrupa múltiples consultas de usuarios en una única llamada a la API cuando sea posible. |

En la práctica, un enfoque por niveles funciona bien: enruta las solicitudes sencillas a un modelo rápido y económico y escala solo las consultas complejas a uno más capaz.


## Resumen

In this lesson you learned how to:

1. **Agregar observabilidad** a las interacciones de los agentes con temporización y registro, sentando las bases para el rastreo y el monitoreo.
2. **Evaluar las respuestas del agente** automáticamente usando un agente evaluador que puntúa completitud, exactitud y utilidad.
3. **Gestionar costos** mediante la optimización de prompts, la selección de modelos, el uso de caché y presupuestos de tokens.

These production patterns help ensure your AI agents are reliable, measurable, and cost-effective at scale.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Descargo de responsabilidad**:
Este documento ha sido traducido utilizando el servicio de traducción automática [Co-op Translator](https://github.com/Azure/co-op-translator). Aunque nos esforzamos por la precisión, tenga en cuenta que las traducciones automatizadas pueden contener errores o inexactitudes. El documento original en su idioma nativo debe considerarse la fuente autorizada. Para información crítica, se recomienda una traducción profesional realizada por un traductor humano. No nos hacemos responsables de ningún malentendido o mala interpretación que surja del uso de esta traducción.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
